# Episode 20: LLM tool calling from scratch: a calculator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep020-tools.ipynb)

We fine-tune the model we pretrained on the web to call a calculator, in plain text, and measure how often it calls it right, on questions asked the way training asks and in new ways, with the calculator and without it.

[The video](https://youtu.be/UrlqjWEuYH4) · [The episode's page](https://mvgpt.nenuadrian.com/ep020-tools/) · [The docs](https://nenuadrian.github.io/mvGPT/tools/) · [The code at `ep020/end`](https://github.com/nenuadrian/mvGPT/tree/ep020/end)

This notebook runs the episode's commands on its code, as the episode left it. Models and runs from the episodes come from [the Hub](https://huggingface.co/nenuadrian/mvGPT-checkpoints), the files the videos used. What you train here comes out a little different from the video's, as training runs for a set time, on another machine. It runs on a GPU runtime, and Colab's free T4 will do.

In [ ]:
# The code as the episode left it, at its last tag, and what it needs.
import os

if not os.path.isdir("/content/mvGPT"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep020/end https://github.com/nenuadrian/mvGPT /content/mvGPT
%cd /content/mvGPT
!pip install --quiet -r requirements.txt
!python setup.py -q build_ext --inplace  # the C++ BPE tokenizer, as a Python module

import torch

print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")

from huggingface_hub import hf_hub_download


def fetch(*names):
    # Models from the episodes, from the Hub: the files the videos used, where the scripts look for them.
    for name in names:
        print(hf_hub_download("nenuadrian/mvGPT-checkpoints", f"checkpoints/{name}/model.pt", local_dir="."))


fetch("pretrain-web")

## The tests
The calculator, what it refuses, and the loop that calls it.

In [ ]:
!python -m pytest -q tests/test_tools.py

## The data
50,000 questions for the calculator, and five held-out sets; some come from web pages.

In [ ]:
!python -c "from data import WEB_SHARDS, download_shard; download_shard(WEB_SHARDS - 1)"  # FineWeb-Edu's last file, which pretraining never read
!python tool_data.py

## The web model, shown a few examples

In [ ]:
!python tool_eval.py checkpoints/pretrain-web/model.pt --shots

## Fine-tuning it to call the calculator
Ten minutes, to `checkpoints/tools`.

In [ ]:
!python tool_sft.py 10
!python tool_eval.py checkpoints/tools/model.pt

## Asking it

In [ ]:
!python tools.py checkpoints/tools/model.pt "What is 48213 * 907?"
!python tools.py checkpoints/tools/model.pt "What is 48213 * 907?" --no-tool
!python tools.py checkpoints/tools/model.pt "A baker makes 144 loaves a day. How many loaves does she make in 6 days?"
!python tools.py checkpoints/tools/model.pt "A café sells 386 cups of coffee a day. How many does it sell in 29 days?"

Next: [episode 21, AI agent from scratch, with subagents](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep021-agent.ipynb)